In [26]:
import glob
import open3d as o3d
import cv2
import matplotlib.pyplot as plt
import numpy as np
from Final_Functions import load_and_undistort_images, CalibrateStereoCamera

# Sequence 1

In [27]:
left_images, right_images, left_images_gray, right_images_gray = load_and_undistort_images("1_no_obstacle")

In [34]:
min_disp = 0
# Search range must be a multiple of 16; here it searches 16 disparity levels
num_disp =  2* 8
# Block matching window size; larger is smoother but loses detail
block_size = 25

# StereoBM finds the best horizontal match in the right image for each block in the left image
stereo = cv2.StereoBM_create(numDisparities = num_disp, blockSize = block_size)
stereo.setMinDisparity(min_disp)          # Same value as min_disp
stereo.setDisp12MaxDiff(1)              # Maximum allowed left-right consistency error
stereo.setUniquenessRatio(25)              # Best match must beat the second best by this percent; higher is stricter
stereo.setSpeckleRange(4)                 # Maximum disparity change inside a connected region
stereo.setSpeckleWindowSize(40)            # Window size for removing small disparity specks
# compute returns fixed-point disparity (x16); divide by 16 to get pixels
disparity = stereo.compute(left_images_gray[0], right_images_gray[0]).astype(np.float32) / 16

In [35]:
calibration = np.load("calibration_maps.npz")
Q = calibration["Q"]

min_disp = 2
max_disp = 20

valid = (
    (disparity > min_disp) &
    (disparity < max_disp)
)

points_3D = cv2.reprojectImageTo3D(disparity, Q)

mask = valid & np.isfinite(points_3D).all(axis=2)

points = points_3D[mask]
colors = left_images[0][mask] / 255.0

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])